In [ ]:
!pip install sec-edgar-downloader yfinance -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 2.1 MB/s eta 0:00:00


In [ ]:
from sec_edgar_downloader import Downloader

# SEC requires every request to identify who's asking (this is a public data courtesy rule, not optional)
dl = Downloader("PersonalProject", "youremail@example.com")  # <-- replace with your real email

tickers = ["WMT", "COST", "TGT", "BBY"]

for ticker in tickers:
    dl.get("10-K", ticker, after="2020-01-01", before="2026-01-01")
    dl.get("10-Q", ticker, after="2020-01-01", before="2026-01-01")

print("Done downloading filings.")

Done downloading filings.


In [ ]:
import os

for root, dirs, files in os.walk("sec-edgar-filings"):
    level = root.replace("sec-edgar-filings", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level > 3:  # don't spam every single file, just confirm structure
        continue
    for f in files[:2]:
        print(f"{indent}  {f}")

sec-edgar-filings/
  COST/
    10-K/
      0000909832-20-000017/
        full-submission.txt
      0000909832-23-000042/
        full-submission.txt
      0000909832-22-000021/
        full-submission.txt
      0000909832-24-000049/
        full-submission.txt
      0000909832-25-000101/
        full-submission.txt
      0000909832-21-000014/
        full-submission.txt
    10-Q/
      0000909832-24-000029/
        full-submission.txt
      0000909832-21-000021/
        full-submission.txt
      0000909832-25-000169/
        full-submission.txt
      0000909832-20-000026/
        full-submission.txt
      0000909832-25-000033/
        full-submission.txt
      0000909832-20-000004/
        full-submission.txt
      0000909832-25-000015/
        full-submission.txt
      0000909832-20-000009/
        full-submission.txt
      0000909832-22-000005/
        full-submission.txt
      0000909832-22-000014/
        full-submission.txt
      0000909832-24-000079/
        full-submission.txt
 

In [ ]:
sample_path = "sec-edgar-filings/WMT/10-K/0000104169-24-000056/full-submission.txt"

with open(sample_path, 'r', encoding='utf-8', errors='ignore') as f:
    content = f.read()

print(f"Total length: {len(content)} characters")
print("\n--- First 2000 characters ---")
print(content[:2000])

Total length: 13085875 characters

--- First 2000 characters ---
<SEC-DOCUMENT>0000104169-24-000056.txt : 20240315
<SEC-HEADER>0000104169-24-000056.hdr.sgml : 20240315
<ACCEPTANCE-DATETIME>20240315165448
ACCESSION NUMBER:		0000104169-24-000056
CONFORMED SUBMISSION TYPE:	10-K
PUBLIC DOCUMENT COUNT:		108
CONFORMED PERIOD OF REPORT:	20240131
FILED AS OF DATE:		20240315
DATE AS OF CHANGE:		20240315

FILER:

	COMPANY DATA:	
		COMPANY CONFORMED NAME:			Walmart Inc.
		CENTRAL INDEX KEY:			0000104169
		STANDARD INDUSTRIAL CLASSIFICATION:	RETAIL-VARIETY STORES [5331]
		ORGANIZATION NAME:           	07 Trade & Services
		IRS NUMBER:				710415188
		STATE OF INCORPORATION:			DE
		FISCAL YEAR END:			0131

	FILING VALUES:
		FORM TYPE:		10-K
		SEC ACT:		1934 Act
		SEC FILE NUMBER:	001-06991
		FILM NUMBER:		24755648

	BUSINESS ADDRESS:	
		STREET 1:		702 SOUTHWEST 8TH ST
		CITY:			BENTONVILLE
		STATE:			AR
		ZIP:			72716
		BUSINESS PHONE:		5012734000

	MAIL ADDRESS:	
		STREET 1:		702 SOUTHWEST 8TH STRE

In [ ]:
import re

def extract_primary_document(filepath):
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()

    # SEC submissions bundle multiple <DOCUMENT> blocks together.
    # We want the one whose <TYPE> matches our filing type (10-K or 10-Q)
    documents = re.split(r'<DOCUMENT>', content)[1:]  # skip header before first <DOCUMENT>

    for doc in documents:
        type_match = re.search(r'<TYPE>(10-K|10-Q)', doc)
        if type_match:
            return doc
    return None

sample_doc = extract_primary_document("sec-edgar-filings/WMT/10-K/0000104169-24-000056/full-submission.txt")

print(f"Extracted document length: {len(sample_doc)} characters")
print(sample_doc[:1000])

Extracted document length: 2323988 characters

<TYPE>10-K
<SEQUENCE>1
<FILENAME>wmt-20240131.htm
<DESCRIPTION>10-K
<TEXT>
<XBRL>
<?xml version='1.0' encoding='ASCII'?>
<html xmlns="http://www.w3.org/1999/xhtml" xmlns:wmt="http://www.walmart.com/20240131" xmlns:us-gaap="http://fasb.org/us-gaap/2023" xmlns:ix="http://www.xbrl.org/2013/inlineXBRL" xmlns:xbrldi="http://xbrl.org/2006/xbrldi" xmlns:link="http://www.xbrl.org/2003/linkbase" xmlns:ixt-sec="http://www.sec.gov/inlineXBRL/transformation/2015-08-31" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:iso4217="http://www.xbrl.org/2003/iso4217" xmlns:ixt="http://www.xbrl.org/inlineXBRL/transformation/2020-02-12" xmlns:country="http://xbrl.sec.gov/country/2023" xmlns:srt="http://fasb.org/srt/2023" xmlns:dei="http://xbrl.sec.gov/dei/2023" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:ecd="http://xbrl.sec.gov/ecd/2023" xmlns:xbrli="http://www.xbrl.org/2003/instance" xml:lang="en-US"><head><meta http-equiv="Content-Type" conte

In [ ]:
from bs4 import BeautifulSoup

def clean_html(raw_doc):
    # Remove the SEC header lines (<TYPE>, <SEQUENCE>, etc.) before the actual HTML starts
    html_start = raw_doc.find('<html')
    if html_start == -1:
        html_start = raw_doc.find('<HTML')
    html_content = raw_doc[html_start:]

    soup = BeautifulSoup(html_content, 'html.parser')

    # Remove script/style tags entirely - they're code, not readable content
    for tag in soup(['script', 'style']):
        tag.decompose()

    text = soup.get_text(separator=' ')

    # Collapse excessive whitespace (HTML often has tons of extra spaces/newlines)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

clean_text = clean_html(sample_doc)
print(f"Clean text length: {len(clean_text)} characters")
print(clean_text[:2000])

Clean text length: 403193 characters
wmt-20240131 0000104169 2024 FY false http://fasb.org/us-gaap/2023#AccountsPayableCurrent http://fasb.org/us-gaap/2023#AccountsPayableCurrent P1Y P5Y 0000104169 2023-02-01 2024-01-31 0000104169 wmt:CommonStockparvalue0.10pershareMember 2023-02-01 2024-01-31 0000104169 wmt:A2.550NotesDue2026Member 2023-02-01 2024-01-31 0000104169 wmt:A1050NotesDue2026Member 2023-02-01 2024-01-31 0000104169 wmt:A1500NotesDue2028Member 2023-02-01 2024-01-31 0000104169 wmt:A4875NotesDue2029Member 2023-02-01 2024-01-31 0000104169 wmt:A5750NotesDue2030Member 2023-02-01 2024-01-31 0000104169 wmt:A1800NotesDue2031Member 2023-02-01 2024-01-31 0000104169 wmt:A5625NotesDue2034Member 2023-02-01 2024-01-31 0000104169 wmt:A5250NotesDue2035Member 2023-02-01 2024-01-31 0000104169 wmt:A4875NotesDue2039Member 2023-02-01 2024-01-31 0000104169 2023-07-31 iso4217:USD 0000104169 2024-03-13 xbrli:shares 0000104169 2022-02-01 2023-01-31 0000104169 2021-02-01 2022-01-31 iso4217:USD xbrli:sh

In [ ]:
def clean_html(raw_doc):
    html_start = raw_doc.find('<html')
    if html_start == -1:
        html_start = raw_doc.find('<HTML')
    html_content = raw_doc[html_start:]

    soup = BeautifulSoup(html_content, 'html.parser')

    # Remove script/style tags
    for tag in soup(['script', 'style']):
        tag.decompose()

    # Remove the inline-XBRL hidden metadata block (ix:header) - not visible to humans, just tagging data
    for tag in soup.find_all(re.compile(r'ix:header', re.I)):
        tag.decompose()

    # Also remove anything explicitly hidden via CSS (display:none is common for XBRL tagging wrappers)
    for tag in soup.find_all(style=re.compile(r'display:\s*none', re.I)):
        tag.decompose()

    text = soup.get_text(separator=' ')
    text = re.sub(r'\s+', ' ', text).strip()

    return text

clean_text = clean_html(sample_doc)
print(f"Clean text length: {len(clean_text)} characters")
print(clean_text[:2000])

Clean text length: 375132 characters
wmt-20240131 UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 ___________________________________________ FORM 10-K ___________________________________________ ☒ Annual report pursuant to section 13 or 15(d) of the Securities Exchange Act of 1934 For the fiscal year ended January 31 , 2024 , or ☐ Transition report pursuant to section 13 or 15(d) of the Securities Exchange Act of 1934 Commission file number 001-06991 . ___________________________________________ WALMART INC. (Exact name of registrant as specified in its charter) ___________________________________________ Delaware 71-0415188 (State or other jurisdiction of incorporation or organization) (IRS Employer Identification No.) 702 S.W. 8th Street 72716 Bentonville, AR (Address of principal executive offices) (Zip Code) Registrant's telephone number, including area code: ( 479 ) 273-4000 Securities registered pursuant to Section 12(b) of the Act: Title of each class Tr

In [ ]:
def extract_mda(clean_text, filing_type="10-K"):
    if filing_type == "10-K":
        start_pattern = r"Item\s+7\.?\s+Management['’]s\s+Discussion"
        end_pattern = r"Item\s+7A\.?\s+Quantitative"
    else:
        start_pattern = r"Item\s+2\.?\s+Management['’]s\s+Discussion"
        end_pattern = r"Item\s+3\.?\s+Quantitative"

    # Find ALL matches, not just the first - the real section is usually the LAST match
    start_matches = list(re.finditer(start_pattern, clean_text, re.IGNORECASE))

    if not start_matches:
        return None

    start_idx = start_matches[-1].start()  # take the last occurrence

    # Now search for the end pattern, but only AFTER our chosen start point
    end_match = re.search(end_pattern, clean_text[start_idx:], re.IGNORECASE)

    if end_match:
        end_idx = start_idx + end_match.start()
    else:
        end_idx = start_idx + 50000

    return clean_text[start_idx:end_idx]

mda_text = extract_mda(clean_text, filing_type="10-K")

if mda_text:
    print(f"MD&A section length: {len(mda_text)} characters")
    print(mda_text[:1500])
else:
    print("Could not find MD&A section - let's debug")

MD&A section length: 60821 characters
ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF OPERATIONS Overview This discussion, which presents our results for the fiscal years ended January 31, 2024 ("fiscal 2024"), January 31, 2023 ("fiscal 2023") and January 31, 2022 ("fiscal 2022"), should be read in conjunction with our Consolidated Financial Statements and the accompanying notes. We intend for this discussion to provide the reader with information that will assist in understanding our financial statements, the changes in certain key items in those financial statements from period to period and the primary factors that accounted for those changes. We also discuss certain performance metrics that management uses to assess the Company's performance. Additionally, the discussion provides information about the financial results of each of the three segments to provide a better understanding of how each of those segments and its results of operations affect

In [ ]:
import os

def get_filing_type_from_path(filepath):
    if "10-K" in filepath:
        return "10-K"
    elif "10-Q" in filepath:
        return "10-Q"
    return None

all_filings = []

base_dir = "sec-edgar-filings"
for ticker in tickers:
    for filing_type in ["10-K", "10-Q"]:
        folder = os.path.join(base_dir, ticker, filing_type)
        if not os.path.exists(folder):
            continue
        for accession in os.listdir(folder):
            filepath = os.path.join(folder, accession, "full-submission.txt")
            if os.path.exists(filepath):
                all_filings.append({
                    'ticker': ticker,
                    'filing_type': filing_type,
                    'accession': accession,
                    'filepath': filepath
                })

print(f"Total filings found: {len(all_filings)}")
print(f"Breakdown by ticker:")
import pandas as pd
filings_df = pd.DataFrame(all_filings)
print(filings_df.groupby(['ticker', 'filing_type']).size())

Total filings found: 96
Breakdown by ticker:
ticker  filing_type
BBY     10-K            6
        10-Q           18
COST    10-K            6
        10-Q           18
TGT     10-K            6
        10-Q           18
WMT     10-K            6
        10-Q           18
dtype: int64


In [ ]:
def process_filing(filepath, filing_type):
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            raw_content = f.read()

        primary_doc = extract_primary_document_from_content(raw_content)
        if primary_doc is None:
            return None, "no_primary_document_found"

        clean = clean_html(primary_doc)
        mda = extract_mda(clean, filing_type=filing_type)

        if mda is None or len(mda) < 500:  # suspiciously short = likely extraction failure
            return None, "mda_not_found_or_too_short"

        return mda, "success"

    except Exception as e:
        return None, f"error: {str(e)}"

# Small adjustment: refactor Cell 5's function to work directly on already-read content
def extract_primary_document_from_content(content):
    documents = re.split(r'<DOCUMENT>', content)[1:]
    for doc in documents:
        type_match = re.search(r'<TYPE>(10-K|10-Q)', doc)
        if type_match:
            return doc
    return None

results = []

for filing in all_filings:
    mda_text, status = process_filing(filing['filepath'], filing['filing_type'])
    results.append({
        'ticker': filing['ticker'],
        'filing_type': filing['filing_type'],
        'accession': filing['accession'],
        'mda_text': mda_text,
        'mda_length': len(mda_text) if mda_text else 0,
        'status': status
    })

results_df = pd.DataFrame(results)

print("Status breakdown:")
print(results_df['status'].value_counts())
print("\nSuccess rate by ticker:")
print(results_df.groupby('ticker')['status'].apply(lambda x: (x == 'success').mean()))

Status breakdown:
status
success                       72
mda_not_found_or_too_short    24
Name: count, dtype: int64

Success rate by ticker:
ticker
BBY     1.0
COST    0.0
TGT     1.0
WMT     1.0
Name: status, dtype: float64


In [ ]:
# Grab one Costco filing and walk through each step manually
costco_filing = [f for f in all_filings if f['ticker'] == 'COST' and f['filing_type'] == '10-K'][0]
print("Debugging:", costco_filing['filepath'])

with open(costco_filing['filepath'], 'r', encoding='utf-8', errors='ignore') as f:
    raw_content = f.read()

primary_doc = extract_primary_document_from_content(raw_content)
print(f"Primary doc found: {primary_doc is not None}")

clean = clean_html(primary_doc)
print(f"Clean text length: {len(clean)}")

# Search for ANY mention of "Management's Discussion" regardless of our strict pattern
matches = [m.start() for m in re.finditer(r"Management['’]s\s+Discussion", clean, re.IGNORECASE)]
print(f"Number of 'Management's Discussion' mentions: {len(matches)}")

if matches:
    # Show context around the LAST mention (likely the real section)
    idx = matches[-1]
    print("\nContext around last mention:")
    print(clean[max(0,idx-200):idx+300])

Debugging: sec-edgar-filings/COST/10-K/0000909832-20-000017/full-submission.txt
Primary doc found: True
Clean text length: 218053
Number of 'Management's Discussion' mentions: 5

Context around last mention:
s to net income relate to net income attributable to Costco. For discussion related to the results of operations and changes in financial condition for 2019 compared to 2018 refer to Part II, Item 7, Management's Discussion and Analysis of Financial Condition and Results of Operations in our fiscal year 2019 Form 10-K, which was filed with the United States Securities and Exchange Commission on October 11, 2019. 22 Table of Contents Highlights for 2020 included: • We opened 16 new warehouses, in


In [ ]:
def extract_mda_v2(clean_text, filing_type="10-K"):
    if filing_type == "10-K":
        # Require "Item 7" to be followed closely by "Management's Discussion"
        # AND make sure we're not matching a cross-reference (look for common giveaway phrases nearby)
        pattern = r"Item\s+7\.?\s*Management['’]s\s+Discussion\s+and\s+Analysis"
        end_pattern = r"Item\s+7A\.?\s*Quantitative"
    else:
        pattern = r"Item\s+2\.?\s*Management['’]s\s+Discussion\s+and\s+Analysis"
        end_pattern = r"Item\s+3\.?\s*Quantitative"

    matches = list(re.finditer(pattern, clean_text, re.IGNORECASE))

    if not matches:
        return None

    # Filter out matches that look like cross-references (nearby words like "refer to", "in our", "filed")
    real_matches = []
    for m in matches:
        context_before = clean_text[max(0, m.start()-100):m.start()].lower()
        if 'refer to' not in context_before and 'in our' not in context_before and 'filed with' not in context_before:
            real_matches.append(m)

    if not real_matches:
        return None

    start_idx = real_matches[-1].start()

    end_match = re.search(end_pattern, clean_text[start_idx:], re.IGNORECASE)
    end_idx = start_idx + end_match.start() if end_match else start_idx + 50000

    return clean_text[start_idx:end_idx]

# Re-test on Costco
mda_test = extract_mda_v2(clean, filing_type="10-K")
if mda_test:
    print(f"Length: {len(mda_test)}")
    print(mda_test[:1000])
else:
    print("Still failing")

Length: 97
Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations 21 


In [ ]:
def extract_mda_v3(clean_text, filing_type="10-K"):
    if filing_type == "10-K":
        pattern = r"Item\s+7\.?\s*Management['’]s\s+Discussion\s+and\s+Analysis"
        end_pattern = r"Item\s+7A\.?\s*Quantitative"
    else:
        pattern = r"Item\s+2\.?\s*Management['’]s\s+Discussion\s+and\s+Analysis"
        end_pattern = r"Item\s+3\.?\s*Quantitative"

    matches = list(re.finditer(pattern, clean_text, re.IGNORECASE))
    if not matches:
        return None

    # Filter out cross-references
    candidates = []
    for m in matches:
        context_before = clean_text[max(0, m.start()-100):m.start()].lower()
        if 'refer to' in context_before or 'in our' in context_before or 'filed with' in context_before:
            continue

        # Check how far away the NEXT "Item 7A" style match is
        end_match = re.search(end_pattern, clean_text[m.start():], re.IGNORECASE)
        gap = end_match.start() if end_match else 999999

        candidates.append((m.start(), gap))

    if not candidates:
        return None

    # A real MD&A section has a LARGE gap to the next item (thousands of characters)
    # A table-of-contents entry has a TINY gap (just the next TOC line)
    real_candidates = [c for c in candidates if c[1] > 2000]

    if not real_candidates:
        return None

    start_idx, gap = real_candidates[-1]  # last genuine candidate
    end_idx = start_idx + gap

    return clean_text[start_idx:end_idx]

mda_test = extract_mda_v3(clean, filing_type="10-K")
if mda_test:
    print(f"Length: {len(mda_test)}")
    print(mda_test[:1000])
else:
    print("Still failing")

Still failing


In [ ]:
pattern = r"Item\s+7\.?\s*Management['’]s\s+Discussion\s+and\s+Analysis"
end_pattern = r"Item\s+7A\.?\s*Quantitative"

matches = list(re.finditer(pattern, clean, re.IGNORECASE))
print(f"Total raw matches: {len(matches)}")

for i, m in enumerate(matches):
    context_before = clean[max(0, m.start()-100):m.start()].lower()
    is_crossref = 'refer to' in context_before or 'in our' in context_before or 'filed with' in context_before

    end_match = re.search(end_pattern, clean[m.start():], re.IGNORECASE)
    gap = end_match.start() if end_match else None

    print(f"\nMatch {i}: position {m.start()}, is_crossref={is_crossref}, gap_to_next_item7A={gap}")
    print(f"  Preview: ...{clean[m.start():m.start()+150]}...")


Total raw matches: 1

Match 0: position 4054, is_crossref=False, gap_to_next_item7A=97
  Preview: ...Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations 21 Item 7A. Quantitative and Qualitative Disclosures Abo...


In [ ]:
matches_loose = list(re.finditer(r"Management['’]s\s+Discussion", clean, re.IGNORECASE))
print(f"Total loose matches: {len(matches_loose)}")

for i, m in enumerate(matches_loose):
    print(f"\n--- Match {i} at position {m.start()} ---")
    print(clean[max(0, m.start()-80):m.start()+200])


Total loose matches: 5

--- Match 0 at position 4062 ---
er Purchases of Equity Securities 18 Item 6. Selected Financial Data 19 Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations 21 Item 7A. Quantitative and Qualitative Disclosures About Market Risk 30 Item 8. Financial Statements and Supplem

--- Match 1 at position 6563 ---
 Item 1A-Risk Factors ”, and other factors noted in the section titled “ Item 7-Management's Discussion and Analysis of Financial Condition and Results of Operations ” and in the consolidated financial statements and related notes in Item 8 of this Report. Forward-looking stateme

--- Match 2 at position 68792 ---
 and key operating metrics. This information should be read in conjunction with Management’s Discussion and Analysis of Financial Condition and Results of Operations, included in Item 7 of this Report, and our consolidated financial statements and notes thereto, included in Item 

--- Match 3 at position 71293 

In [ ]:
def extract_mda_v4(clean_text, filing_type="10-K"):
    if filing_type == "10-K":
        # Allow period, dash (-, –, —), or colon between "Item 7" and "Management's"
        pattern = r"Item\s+7[\.\-–—:]?\s*Management['’]s\s+Discussion\s+and\s+Analysis"
        end_pattern = r"Item\s+7A[\.\-–—:]?\s*Quantitative"
    else:
        pattern = r"Item\s+2[\.\-–—:]?\s*Management['’]s\s+Discussion\s+and\s+Analysis"
        end_pattern = r"Item\s+3[\.\-–—:]?\s*Quantitative"

    matches = list(re.finditer(pattern, clean_text, re.IGNORECASE))
    if not matches:
        return None

    candidates = []
    for m in matches:
        context_before = clean_text[max(0, m.start()-100):m.start()].lower()
        if 'refer to' in context_before or 'in our' in context_before or 'filed with' in context_before:
            continue
        end_match = re.search(end_pattern, clean_text[m.start():], re.IGNORECASE)
        gap = end_match.start() if end_match else 999999
        candidates.append((m.start(), gap))

    real_candidates = [c for c in candidates if c[1] > 2000]
    if not real_candidates:
        return None

    start_idx, gap = real_candidates[-1]
    end_idx = start_idx + gap

    return clean_text[start_idx:end_idx]

mda_test = extract_mda_v4(clean, filing_type="10-K")
if mda_test:
    print(f"Length: {len(mda_test)}")
    print(mda_test[:1000])
else:
    print("Still failing")

Length: 33541
Item 7—Management's Discussion and Analysis of Financial Conditions and Results of Operations (amounts in millions, except per share, share, membership fee, and warehouse count data) Overview We believe that the most important driver of our profitability is increasing net sales, particularly comparable sales growth. Net sales includes our core merchandise categories (food and sundries, hardlines, softlines, and fresh foods), warehouse ancillary and other businesses. We define comparable sales as net sales from warehouses open for more than one year, including remodels, relocations and expansions, and sales related to e-commerce websites operating for more than one year. Comparable sales growth is achieved through increasing shopping frequency from new and existing members and the amount they spend on each visit (average ticket). Sales comparisons can also be particularly influenced by certain factors that are beyond our control: fluctuations in currency exchange rates (wi

In [ ]:
def process_filing_v2(filepath, filing_type):
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            raw_content = f.read()

        primary_doc = extract_primary_document_from_content(raw_content)
        if primary_doc is None:
            return None, "no_primary_document_found"

        clean_txt = clean_html(primary_doc)
        mda = extract_mda_v4(clean_txt, filing_type=filing_type)

        if mda is None or len(mda) < 500:
            return None, "mda_not_found_or_too_short"

        return mda, "success"

    except Exception as e:
        return None, f"error: {str(e)}"

results = []
for filing in all_filings:
    mda_text, status = process_filing_v2(filing['filepath'], filing['filing_type'])
    results.append({
        'ticker': filing['ticker'],
        'filing_type': filing['filing_type'],
        'accession': filing['accession'],
        'mda_text': mda_text,
        'mda_length': len(mda_text) if mda_text else 0,
        'status': status
    })

results_df = pd.DataFrame(results)

print("Status breakdown:")
print(results_df['status'].value_counts())
print("\nSuccess rate by ticker:")
print(results_df.groupby('ticker')['status'].apply(lambda x: (x == 'success').mean()))



Status breakdown:
status
success                       95
mda_not_found_or_too_short     1
Name: count, dtype: int64

Success rate by ticker:
ticker
BBY     0.958333
COST    1.000000
TGT     1.000000
WMT     1.000000
Name: status, dtype: float64


In [ ]:
failed = results_df[results_df['status'] != 'success']
print(failed)

   ticker filing_type             accession mda_text  mda_length  \
89    BBY        10-Q  0000764478-21-000068     None           0   

                        status  
89  mda_not_found_or_too_short  


In [ ]:
bby_filepath = "sec-edgar-filings/BBY/10-Q/0000764478-21-000068/full-submission.txt"

with open(bby_filepath, 'r', encoding='utf-8', errors='ignore') as f:
    raw_content = f.read()

primary_doc = extract_primary_document_from_content(raw_content)
print(f"Primary doc found: {primary_doc is not None}")

if primary_doc:
    clean_bby = clean_html(primary_doc)
    print(f"Clean text length: {len(clean_bby)}")

    # Loose search for any "Management's Discussion" mention (10-Q uses Item 2, not Item 7)
    matches_loose = list(re.finditer(r"Management['’]s\s+Discussion", clean_bby, re.IGNORECASE))
    print(f"Loose matches found: {len(matches_loose)}")

    for i, m in enumerate(matches_loose):
        print(f"\n--- Match {i} at position {m.start()} ---")
        print(clean_bby[max(0, m.start()-80):m.start()+200])

Primary doc found: True
Clean text length: 91068
Loose matches found: 4

--- Match 0 at position 3552 ---
er 31, 2020 7 f) Notes to Condensed Consolidated Financial Statements 8 Item 2. Management’s Discussion and Analysis of Financial Condition and Results of Operations 15 Item 3. Quantitative and Qualitative Disclosures About Market Risk 25 Item 4. Controls and Procedures 25 Part I

--- Match 1 at position 34324 ---
ts beginning in the fourth quarter of fiscal 2022. 14 Table of Contents Item 2. Management's Discussion and Analysis of Financial Condition and Results of Operations Unless the context otherwise requires, the use of the terms “Best Buy,” “we,” “us” and “our” refers to Best Buy Co

--- Match 2 at position 34698 ---
titute incorporation by reference of the information contained on the websites. Management’s Discussion and Analysis of Financial Condition and Results of Operations (“MD&A”) is intended to provide a reader of our financial statements with a narrative from the p

In [ ]:
mda_bby = extract_mda_v4(clean_bby, filing_type="10-Q")

if mda_bby:
    print(f"Success! Length: {len(mda_bby)}")
    print(mda_bby[:500])
else:
    print("Failed - let's see why, step by step")

    pattern = r"Item\s+2[\.\-–—:]?\s*Management['’]s\s+Discussion\s+and\s+Analysis"
    end_pattern = r"Item\s+3[\.\-–—:]?\s*Quantitative"

    matches = list(re.finditer(pattern, clean_bby, re.IGNORECASE))
    print(f"Raw pattern matches: {len(matches)}")

    for m in matches:
        context_before = clean_bby[max(0, m.start()-100):m.start()].lower()
        is_crossref = 'refer to' in context_before or 'in our' in context_before or 'filed with' in context_before

        end_match = re.search(end_pattern, clean_bby[m.start():], re.IGNORECASE)
        gap = end_match.start() if end_match else None

        print(f"\nPosition {m.start()}: is_crossref={is_crossref}, gap={gap}")
        print(f"  Context before: ...{context_before}")

Failed - let's see why, step by step
Raw pattern matches: 2

Position 3544: is_crossref=False, gap=97
  Context before: ... october 30, 2021, and october 31, 2020 7 f) notes to condensed consolidated financial statements 8 

Position 34316: is_crossref=True, gap=48247
  Context before: ...ed in our consolidated results beginning in the fourth quarter of fiscal 2022. 14 table of contents 


In [ ]:
def extract_mda_v5(clean_text, filing_type="10-K"):
    if filing_type == "10-K":
        pattern = r"Item\s+7[\.\-–—:]?\s*Management['’]s\s+Discussion\s+and\s+Analysis"
        end_pattern = r"Item\s+7A[\.\-–—:]?\s*Quantitative"
    else:
        pattern = r"Item\s+2[\.\-–—:]?\s*Management['’]s\s+Discussion\s+and\s+Analysis"
        end_pattern = r"Item\s+3[\.\-–—:]?\s*Quantitative"

    matches = list(re.finditer(pattern, clean_text, re.IGNORECASE))
    if not matches:
        return None

    candidates = []
    for m in matches:
        # Look at a WIDER window, but require "Form 10-K" to ALSO appear nearby -
        # that's the real signature of a genuine cross-reference, not just "in our"/"refer to" alone
        context_window = clean_text[max(0, m.start()-150):m.start()+150].lower()
        is_crossref = ('form 10-k' in context_window or 'form 10-q' in context_window) and \
                      ('refer to' in context_window or 'filed with' in context_window or 'in our annual report' in context_window)

        if is_crossref:
            continue

        end_match = re.search(end_pattern, clean_text[m.start():], re.IGNORECASE)
        gap = end_match.start() if end_match else 999999
        candidates.append((m.start(), gap))

    real_candidates = [c for c in candidates if c[1] > 2000]
    if not real_candidates:
        return None

    start_idx, gap = real_candidates[-1]
    end_idx = start_idx + gap

    return clean_text[start_idx:end_idx]

mda_bby = extract_mda_v5(clean_bby, filing_type="10-Q")
if mda_bby:
    print(f"Success! Length: {len(mda_bby)}")
    print(mda_bby[:500])
else:
    print("Still failing")

Success! Length: 48247
Item 2. Management's Discussion and Analysis of Financial Condition and Results of Operations Unless the context otherwise requires, the use of the terms “Best Buy,” “we,” “us” and “our” refers to Best Buy Co., Inc. and its consolidated subsidiaries. Any references to our website addresses do not constitute incorporation by reference of the information contained on the websites. Management’s Discussion and Analysis of Financial Condition and Results of Operations (“MD&A”) is intended to provide 


In [ ]:
def process_filing_v3(filepath, filing_type):
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            raw_content = f.read()

        primary_doc = extract_primary_document_from_content(raw_content)
        if primary_doc is None:
            return None, "no_primary_document_found"

        clean_txt = clean_html(primary_doc)
        mda = extract_mda_v5(clean_txt, filing_type=filing_type)

        if mda is None or len(mda) < 500:
            return None, "mda_not_found_or_too_short"

        return mda, "success"

    except Exception as e:
        return None, f"error: {str(e)}"

results = []
for filing in all_filings:
    mda_text, status = process_filing_v3(filing['filepath'], filing['filing_type'])
    results.append({
        'ticker': filing['ticker'],
        'filing_type': filing['filing_type'],
        'accession': filing['accession'],
        'mda_text': mda_text,
        'mda_length': len(mda_text) if mda_text else 0,
        'status': status
    })

results_df = pd.DataFrame(results)

print("Status breakdown:")
print(results_df['status'].value_counts())
print("\nSuccess rate by ticker:")
print(results_df.groupby('ticker')['status'].apply(lambda x: (x == 'success').mean()))

Status breakdown:
status
success    96
Name: count, dtype: int64

Success rate by ticker:
ticker
BBY     1.0
COST    1.0
TGT     1.0
WMT     1.0
Name: status, dtype: float64


In [ ]:
# We still need the actual filing DATE for each one - let's pull that from the raw file header
def get_filing_date(filepath):
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        header = f.read(2000)  # date info is always near the very top, no need to read the whole file
    match = re.search(r"FILED AS OF DATE:\s*(\d{8})", header)
    if match:
        date_str = match.group(1)
        return pd.to_datetime(date_str, format='%Y%m%d')
    return None

all_filings_df = pd.DataFrame(all_filings)
results_df['filing_date'] = [get_filing_date(f['filepath']) for f in all_filings]

# Merge ticker/date info with our extracted MD&A text
final_df = results_df[results_df['status'] == 'success'].copy()
final_df = final_df[['ticker', 'filing_type', 'filing_date', 'mda_text', 'mda_length']]
final_df = final_df.sort_values(['ticker', 'filing_date']).reset_index(drop=True)

print(final_df.shape)
final_df.head(10)

# Save it so we never have to redo this extraction again
final_df.to_csv('mda_sections.csv', index=False)
print("\nSaved to mda_sections.csv")

(96, 5)

Saved to mda_sections.csv


In [ ]:
!pip install gdown -q

import gdown

file_id = "1iq2RUf8qGFEAk1g8wQntP3habOnR3fXF"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "LM_dictionary.csv", quiet=False)

lm_dict = pd.read_csv("LM_dictionary.csv")
print(lm_dict.shape)
print(lm_dict.columns.tolist())
lm_dict.head()

Downloading...
From: https://drive.google.com/uc?id=1iq2RUf8qGFEAk1g8wQntP3habOnR3fXF
To: /content/LM_dictionary.csv
100%|██████████| 9.09M/9.09M [00:00<00:00, 33.5MB/s]


(86553, 17)
['Word', 'Seq_num', 'Word Count', 'Word Proportion', 'Average Proportion', 'Std Dev', 'Doc Count', 'Negative', 'Positive', 'Uncertainty', 'Litigious', 'Strong_Modal', 'Weak_Modal', 'Constraining', 'Complexity', 'Syllables', 'Source']


,Word,Seq_num,Word Count,Word Proportion,Average Proportion,Std Dev,Doc Count,Negative,Positive,Uncertainty,Litigious,Strong_Modal,Weak_Modal,Constraining,Complexity,Syllables,Source
0,AARDVARK,1,814,3.085383e-08,2.022181e-08,4.059339e-06,158,0,0,0,0,0,0,0,0,2,12of12inf
1,AARDVARKS,2,3,1.137119e-10,7.898767e-12,8.829342e-09,1,0,0,0,0,0,0,0,0,2,12of12inf
2,ABACI,3,9,3.411357e-10,1.067549e-10,5.054031e-08,7,0,0,0,0,0,0,0,0,3,12of12inf
3,ABACK,4,29,1.099215e-09,6.073925e-10,1.523804e-07,28,0,0,0,0,0,0,0,0,2,12of12inf
4,ABACUS,5,9921,3.760453e-07,3.836730e-07,3.392121e-05,1355,0,0,0,0,0,0,0,0,3,12of12inf


In [ ]:
# Words are stored in ALL CAPS in this dictionary - note that for later
positive_words = set(lm_dict[lm_dict['Positive'] != 0]['Word'])
negative_words = set(lm_dict[lm_dict['Negative'] != 0]['Word'])

print(f"Positive words: {len(positive_words)}")
print(f"Negative words: {len(negative_words)}")
print("\nSample positive words:", list(positive_words)[:10])
print("Sample negative words:", list(negative_words)[:10])

def score_sentiment(text):
    # Split into words, uppercase to match dictionary's format, strip punctuation
    words = re.findall(r"[A-Za-z]+", text.upper())
    total_words = len(words)

    if total_words == 0:
        return 0, 0, 0, 0

    pos_count = sum(1 for w in words if w in positive_words)
    neg_count = sum(1 for w in words if w in negative_words)

    net_sentiment = (pos_count - neg_count) / total_words

    return pos_count, neg_count, total_words, net_sentiment

# Test on one filing
sample_mda = final_df.iloc[0]['mda_text']
pos, neg, total, sentiment = score_sentiment(sample_mda)
print(f"\nTest filing: {final_df.iloc[0]['ticker']}, {final_df.iloc[0]['filing_date']}")
print(f"Positive words: {pos}, Negative words: {neg}, Total words: {total}")
print(f"Net sentiment score: {sentiment:.5f}")


Positive words: 354
Negative words: 2355

Sample positive words: ['IMPROVEMENT', 'BEAUTIFULLY', 'BENEFIT', 'SUCCEEDS', 'PLENTIFUL', 'ENHANCEMENT', 'TREMENDOUSLY', 'BOLSTERS', 'INNOVATE', 'EXCLUSIVES']
Sample negative words: ['LAGS', 'NONPRODUCTIVE', 'THREATS', 'SEIZING', 'MISMANAGE', 'VOLATILE', 'HAMPERING', 'PRECLUDE', 'HARSHEST', 'CONSPIRATORS']

Test filing: BBY, 2020-03-23 00:00:00
Positive words: 98, Negative words: 77, Total words: 7914
Net sentiment score: 0.00265


In [ ]:
final_df[['pos_count', 'neg_count', 'total_words', 'sentiment_score']] = final_df['mda_text'].apply(
    lambda text: pd.Series(score_sentiment(text))
)

print(final_df[['ticker', 'filing_date', 'sentiment_score']].head(10))
print("\nSentiment score summary stats:")
print(final_df['sentiment_score'].describe())

# Save this enriched dataset
final_df.to_csv('mda_with_sentiment.csv', index=False)
print("\nSaved to mda_with_sentiment.csv")

  ticker filing_date  sentiment_score
0    BBY  2020-03-23         0.002654
1    BBY  2020-05-27        -0.003660
2    BBY  2020-08-31        -0.001809
3    BBY  2020-11-30        -0.003268
4    BBY  2021-03-19        -0.001524
5    BBY  2021-06-04        -0.000496
6    BBY  2021-08-31        -0.001443
7    BBY  2021-12-03        -0.004270
8    BBY  2022-03-18        -0.001249
9    BBY  2022-06-02        -0.003239

Sentiment score summary stats:
count    96.000000
mean     -0.000027
std       0.002890
min      -0.005962
25%      -0.002163
50%      -0.000113
75%       0.001765
max       0.007304
Name: sentiment_score, dtype: float64

Saved to mda_with_sentiment.csv


In [ ]:
final_df[['pos_count', 'neg_count', 'total_words', 'sentiment_score']] = final_df['mda_text'].apply(
    lambda text: pd.Series(score_sentiment(text))
)

print(final_df[['ticker', 'filing_date', 'sentiment_score']].head(10))
print("\nSentiment score summary stats:")
print(final_df['sentiment_score'].describe())

# Save this enriched dataset
final_df.to_csv('mda_with_sentiment.csv', index=False)
print("\nSaved to mda_with_sentiment.csv")

  ticker filing_date  sentiment_score
0    BBY  2020-03-23         0.002654
1    BBY  2020-05-27        -0.003660
2    BBY  2020-08-31        -0.001809
3    BBY  2020-11-30        -0.003268
4    BBY  2021-03-19        -0.001524
5    BBY  2021-06-04        -0.000496
6    BBY  2021-08-31        -0.001443
7    BBY  2021-12-03        -0.004270
8    BBY  2022-03-18        -0.001249
9    BBY  2022-06-02        -0.003239

Sentiment score summary stats:
count    96.000000
mean     -0.000027
std       0.002890
min      -0.005962
25%      -0.002163
50%      -0.000113
75%       0.001765
max       0.007304
Name: sentiment_score, dtype: float64

Saved to mda_with_sentiment.csv


In [ ]:
import yfinance as yf

price_data = {}
for ticker in tickers:
    df = yf.download(ticker, start="2019-12-01", end="2026-01-01", progress=False)
    price_data[ticker] = df
    print(f"{ticker}: {df.shape[0]} trading days pulled")

/tmp/ipykernel_1976/353803308.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start="2019-12-01", end="2026-01-01", progress=False)


WMT: 1529 trading days pulled


/tmp/ipykernel_1976/353803308.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start="2019-12-01", end="2026-01-01", progress=False)


COST: 1529 trading days pulled


/tmp/ipykernel_1976/353803308.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start="2019-12-01", end="2026-01-01", progress=False)


TGT: 1529 trading days pulled


/tmp/ipykernel_1976/353803308.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start="2019-12-01", end="2026-01-01", progress=False)


BBY: 1529 trading days pulled


In [ ]:
def get_return(ticker_df, filing_date, days_after=5):
    ticker_df = ticker_df.sort_index()
    future_dates = ticker_df.index[ticker_df.index >= filing_date]

    if len(future_dates) < days_after + 1:
        return None

    start_date = future_dates[0]
    end_date = future_dates[days_after]

    start_price = ticker_df.loc[start_date, 'Close']
    end_price = ticker_df.loc[end_date, 'Close']

    # .item() extracts the plain Python number from a single-value pandas object
    pct_return = ((end_price - start_price) / start_price)
    if hasattr(pct_return, 'item'):
        pct_return = pct_return.item()

    return pct_return

test_row = final_df.iloc[0]
test_return = get_return(price_data[test_row['ticker']], test_row['filing_date'], days_after=5)
print(f"Ticker: {test_row['ticker']}, Filing date: {test_row['filing_date']}")
print(f"5-day return after filing: {test_return:.4f} ({test_return*100:.2f}%)")

Ticker: BBY, Filing date: 2020-03-23 00:00:00
5-day return after filing: 0.1787 (17.87%)


In [ ]:
def get_return_safe(ticker, filing_date, days_after):
    try:
        return get_return(price_data[ticker], filing_date, days_after=days_after)
    except Exception:
        return None

final_df['return_5d'] = final_df.apply(
    lambda row: get_return_safe(row['ticker'], row['filing_date'], 5), axis=1
)
final_df['return_10d'] = final_df.apply(
    lambda row: get_return_safe(row['ticker'], row['filing_date'], 10), axis=1
)
final_df['return_20d'] = final_df.apply(
    lambda row: get_return_safe(row['ticker'], row['filing_date'], 20), axis=1
)

print(f"Missing 5d returns: {final_df['return_5d'].isna().sum()}")
print(f"Missing 10d returns: {final_df['return_10d'].isna().sum()}")
print(f"Missing 20d returns: {final_df['return_20d'].isna().sum()}")

final_df[['ticker', 'filing_date', 'sentiment_score', 'return_5d', 'return_10d', 'return_20d']].head(10)

Missing 5d returns: 0
Missing 10d returns: 1
Missing 20d returns: 3


,ticker,filing_date,sentiment_score,return_5d,return_10d,return_20d
0,BBY,2020-03-23,0.002654,0.178734,0.186625,0.337542
1,BBY,2020-05-27,-0.003660,0.039707,0.040140,0.067130
2,BBY,2020-08-31,-0.001809,-0.050942,-0.021574,-0.002361
3,BBY,2020-11-30,-0.003268,-0.055791,-0.064000,-0.068806
4,BBY,2021-03-19,-0.001524,0.009053,-0.001692,0.005500
5,BBY,2021-06-04,-0.000496,-0.013277,-0.066304,0.003270
6,BBY,2021-08-31,-0.001443,-0.047378,-0.056207,-0.054825
7,BBY,2021-12-03,-0.004270,-0.019363,-0.048748,-0.028873
8,BBY,2022-03-18,-0.001249,-0.045944,-0.104594,-0.073882
9,BBY,2022-06-02,-0.003239,-0.077898,-0.146933,-0.168235


In [ ]:
spy_df = yf.download("SPY", start="2019-12-01", end="2026-01-01", progress=False)
print(f"SPY: {spy_df.shape[0]} trading days pulled")

def get_return_safe_spy(filing_date, days_after):
    try:
        return get_return(spy_df, filing_date, days_after=days_after)
    except Exception:
        return None

final_df['spy_return_5d'] = final_df.apply(
    lambda row: get_return_safe_spy(row['filing_date'], 5), axis=1
)
final_df['spy_return_10d'] = final_df.apply(
    lambda row: get_return_safe_spy(row['filing_date'], 10), axis=1
)
final_df['spy_return_20d'] = final_df.apply(
    lambda row: get_return_safe_spy(row['filing_date'], 20), axis=1
)

# Abnormal return = company's return MINUS what the market did over the same period
final_df['abnormal_return_5d'] = final_df['return_5d'] - final_df['spy_return_5d']
final_df['abnormal_return_10d'] = final_df['return_10d'] - final_df['spy_return_10d']
final_df['abnormal_return_20d'] = final_df['return_20d'] - final_df['spy_return_20d']

final_df[['ticker', 'filing_date', 'return_5d', 'spy_return_5d', 'abnormal_return_5d']].head(10)

/tmp/ipykernel_1976/849475246.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  spy_df = yf.download("SPY", start="2019-12-01", end="2026-01-01", progress=False)


SPY: 1529 trading days pulled


,ticker,filing_date,return_5d,spy_return_5d,abnormal_return_5d
0,BBY,2020-03-23,0.178734,0.173581,0.005152
1,BBY,2020-05-27,0.039707,0.028498,0.011209
2,BBY,2020-08-31,-0.050942,-0.046091,-0.004852
3,BBY,2020-11-30,-0.055791,0.019417,-0.075207
4,BBY,2021-03-19,0.009053,0.016689,-0.007636
5,BBY,2021-06-04,-0.013277,0.004046,-0.017323
6,BBY,2021-08-31,-0.047378,-0.001440,-0.045938
7,BBY,2021-12-03,-0.019363,0.038199,-0.057562
8,BBY,2022-03-18,-0.045944,0.018379,-0.064324
9,BBY,2022-06-02,-0.077898,-0.038214,-0.039685


In [ ]:
from scipy import stats

for window in ['5d', '10d', '20d']:
    valid = final_df.dropna(subset=['sentiment_score', f'abnormal_return_{window}'])
    corr, p_value = stats.pearsonr(valid['sentiment_score'], valid[f'abnormal_return_{window}'])
    print(f"{window} window: correlation = {corr:.4f}, p-value = {p_value:.4f}, n = {len(valid)}")

5d window: correlation = 0.1192, p-value = 0.2473, n = 96
10d window: correlation = 0.1513, p-value = 0.1432, n = 95
20d window: correlation = 0.1221, p-value = 0.2438, n = 93


In [ ]:
export_df = final_df[['ticker', 'filing_type', 'filing_date', 'sentiment_score',
                        'pos_count', 'neg_count', 'total_words',
                        'return_5d', 'return_10d', 'return_20d',
                        'abnormal_return_5d', 'abnormal_return_10d', 'abnormal_return_20d']].copy()

export_df.to_csv('dashboard_data.csv', index=False)

from google.colab import files
files.download('dashboard_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>